[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

# UofT FASE ML Bootcamp
#### Friday June 11, 2026
#### TabPFN - Lab 2, Day 4
#### Teaching team: Eldan Cohen, Alex Olson, Hriday Chheda
##### Lab author: Hriday Chheda

#### TabPFN: Tabular Foundation Models


In this lab, you will use **TabPFN** on a small tabular classification dataset.

The goal is to understand how a tabular foundation model is used, and how it differs from a standard supervised learning model.

In particular, you will focus on:

1. Loading and inspecting a tabular dataset
2. Training simple baseline models
3. Running TabPFN on the same data
4. Comparing TabPFN to standard tabular ML baselines


---

We start by installing and importing the required libraries.

In [ ]:
! pip install -q scikit-learn pandas numpy matplotlib seaborn torch "tabpfn==2.0.7"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    classification_report,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

import torch
from tabpfn import TabPFNClassifier

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

# 1. What is TabPFN?

Traditional tabular machine learning usually works like this:

```text
Training data -> Model fitting / optimization -> Trained model -> Predictions
```

TabPFN is different. It is a **pretrained tabular foundation model**.

At a high level, TabPFN receives a small labelled training dataset and a set of test points, and directly predicts labels for the test points:

```text
Training table + training labels + test rows -> TabPFN -> predicted labels/probabilities
```

This is closer to **in-context learning** for tables.

Instead of training a new model from scratch for every dataset, TabPFN uses a pretrained transformer that has already learned how to solve many synthetic tabular prediction problems.

In this lab, we will use TabPFN as a classifier and inspect the full prediction workflow.

# 2. Load a tabular dataset

We will use the **Breast Cancer Wisconsin** dataset from scikit-learn.

Each row is a tumour sample.

Each column is a numerical feature computed from a digitized image of a fine needle aspirate of a breast mass.

The prediction task is binary classification:

```text
malignant vs benign
```

In [ ]:
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

target_names = data.target_names

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target names:", target_names)

Let us inspect the first few rows.

In [ ]:
X.head()

Now inspect the target variable.

In [ ]:
target_df = pd.DataFrame({
    "target_value": y,
    "target_name": [target_names[i] for i in y]
})

target_df.head()

## 2.1 Class balance

Before fitting any model, we should inspect the class distribution.

A model can look artificially good if one class is much more common than the other.

In [ ]:
class_counts = target_df["target_name"].value_counts()
class_percentages = target_df["target_name"].value_counts(normalize=True) * 100

class_balance_df = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages.round(2)
})

class_balance_df

In [ ]:
class_counts.plot(kind="bar")
plt.title("Class distribution")
plt.ylabel("Number of examples")
plt.xlabel("Class")
plt.xticks(rotation=0)
plt.show()

## 2.2 Exercise

Inspect the feature columns.

Which features seem like they might be predictive?

In [ ]:
# TODO: Print all feature names.
# Hint: use X.columns

In [ ]:
# TODO: Create summary statistics for all features.
# Hint: use X.describe()

# 3. Train/test split

We will split the dataset into training and test sets.

The training set is the labelled context that TabPFN will condition on.

The test set is used to evaluate predictions.

In [ ]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Let us inspect the exact training table that will be given to the models.

In [ ]:
X_train.head()

And the corresponding labels.

In [ ]:
pd.DataFrame({
    "target_value": y_train[:10].to_numpy(),
    "target_name": [target_names[i] for i in y_train[:10]]
})

# 4. Build standard ML baselines

Before using TabPFN, we should compare against familiar models.

We will use:

1. Logistic regression
2. Random forest

These are not necessarily tuned. They are simple reference models.

In [ ]:
baseline_models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE
    )
}

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = None

    results = {
        "Train accuracy": accuracy_score(y_train, y_train_pred),
        "Test accuracy": accuracy_score(y_test, y_pred),
    }

    return results, y_pred, y_proba

In [ ]:
baseline_results = {}

for name, model in baseline_models.items():
    results, y_pred, y_proba = evaluate_classifier(
        model,
        X_train,
        y_train,
        X_test,
        y_test
    )

    baseline_results[name] = results

pd.DataFrame(baseline_results).T.round(4)

# 5. Run TabPFN locally

Now we will use the open-source **TabPFN v2** package locally.

This version does **not** require an API key. It runs through the Python package directly.

The first time it is used, it may download pretrained model weights. After that, the weights are cached.

TabPFN follows the scikit-learn style:

```python
clf = TabPFNClassifier()
clf.fit(X_train, y_train)
clf.predict(X_test)
clf.predict_proba(X_test)
```

However, conceptually, its `fit` step is not the same as training logistic regression or a random forest from scratch.

The pretrained TabPFN model uses the training table as context for inference.

In [ ]:
def make_tabpfn_classifier(random_state=RANDOM_STATE):
    """Create a TabPFN classifier in a version-tolerant way."""
    try:
        return TabPFNClassifier(
            device=DEVICE,
            random_state=random_state,
        )
    except TypeError:
        # Some TabPFN versions do not expose random_state in the constructor.
        return TabPFNClassifier(device=DEVICE)


tabpfn_model = make_tabpfn_classifier(random_state=RANDOM_STATE)

tabpfn_model.fit(X_train, y_train)

tabpfn_pred = tabpfn_model.predict(X_test)
tabpfn_proba = tabpfn_model.predict_proba(X_test)[:, 1]

tabpfn_results = {
    "Train accuracy": accuracy_score(y_train, tabpfn_model.predict(X_train)),
    "Test accuracy": accuracy_score(y_test, tabpfn_pred),
}

pd.DataFrame({"TabPFN": tabpfn_results}).T.round(4)

## 5.1 Optional note: API version

Some newer TabPFN workflows also provide an API-based client. We are **not** using that in this lab.

For this bootcamp lab, we use the local open-source TabPFN v2 package so that students can run the notebook without an API key.

The important modelling workflow is the same:

```python
# Local open-source package used in this lab
from tabpfn import TabPFNClassifier

clf = TabPFNClassifier(device=DEVICE)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)
```


# 6. Compare all models

Now combine TabPFN with the baseline models.

In [ ]:
all_results = dict(baseline_results)
all_results["TabPFN"] = tabpfn_results

results_df = pd.DataFrame(all_results).T
results_df.round(4)

## 6.1 Exercise

Which model performs best on this train/test split?


# 7. Inspect TabPFN predictions

A classifier does not only output labels.

It can also output probabilities.

For binary classification, TabPFN returns:

```text
P(class 0), P(class 1)
```

In this dataset:

```text
0 = malignant
1 = benign
```

In [ ]:
prediction_df = X_test.copy()
prediction_df["true_label"] = y_test.to_numpy()
prediction_df["true_name"] = [target_names[i] for i in y_test]
prediction_df["predicted_label"] = tabpfn_pred
prediction_df["predicted_name"] = [target_names[i] for i in tabpfn_pred]
prediction_df["prob_benign"] = tabpfn_proba
prediction_df["prob_malignant"] = 1 - tabpfn_proba
prediction_df["correct"] = prediction_df["true_label"] == prediction_df["predicted_label"]

prediction_df[
    ["true_name", "predicted_name", "prob_malignant", "prob_benign", "correct"]
].head(10)

# 8. Confusion matrix and classification report

The confusion matrix shows which kinds of mistakes the model makes.

In [ ]:
cm = confusion_matrix(y_test, tabpfn_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in target_names],
    columns=[f"pred_{name}" for name in target_names]
)

cm_df

# 9. How does training set size affect TabPFN?

One reason TabPFN is interesting is that it is designed for small tabular datasets.

Let us test how the model behaves when we give it fewer labelled examples.

We will compare TabPFN for different training set sizes.

In [ ]:
def sample_stratified_training_set(X_train, y_train, n_samples, random_state):
    # If we request the full training set, return it directly.
    if n_samples >= len(X_train):
        return X_train.copy(), y_train.copy()

    temp_df = X_train.copy()
    temp_df["target"] = y_train.to_numpy()

    sampled_df, _ = train_test_split(
        temp_df,
        train_size=n_samples,
        random_state=random_state,
        stratify=temp_df["target"]
    )

    X_sample = sampled_df.drop(columns=["target"])
    y_sample = sampled_df["target"]

    return X_sample, y_sample

In [ ]:
def evaluate_models_for_training_size(n_samples, random_state=42):
    X_sample, y_sample = sample_stratified_training_set(
        X_train,
        y_train,
        n_samples=n_samples,
        random_state=random_state
    )

    rows = []

    tabpfn = make_tabpfn_classifier(random_state=random_state)

    tabpfn.fit(X_sample, y_sample)
    pred = tabpfn.predict(X_test)
    proba = tabpfn.predict_proba(X_test)[:, 1]

    rows.append({
        "model": "TabPFN",
        "n_train": n_samples,
        "Train accuracy": accuracy_score(y_sample, tabpfn.predict(X_sample)),
        "Test accuracy": accuracy_score(y_test, pred),
    })

    return rows

In [ ]:
training_sizes = [20, 50, 100, 200, len(X_train)]

training_size_rows = []

for n in training_sizes:
    print(f"Evaluating n_train={n}")
    training_size_rows.extend(
        evaluate_models_for_training_size(
            n_samples=n,
            random_state=RANDOM_STATE
        )
    )

training_size_df = pd.DataFrame(training_size_rows)
training_size_df

## 9.1 Exercise

What happens when the training set is very small?

Does TabPFN remain competitive?


# 10. What should we take away?

In this lab, you saw that TabPFN can be used like a scikit-learn classifier, but it represents a different modelling idea.

A standard model learns parameters for this specific dataset.

TabPFN uses a pretrained foundation model and conditions on the training table to make predictions.

Important lessons:

1. TabPFN is designed for small tabular datasets.
2. The local open-source package can be used.
3. It can work well with little or no hyperparameter tuning.
4. The train/test split still matters.
5. Predicted probabilities help us inspect confidence and uncertainty.

TabPFN is not magic. It is another modelling tool, and it should be evaluated carefully like any other ML method.

# 12. Optional extension

Try replacing the dataset with another scikit-learn tabular dataset.

Examples:

```python
from sklearn.datasets import load_wine
from sklearn.datasets import load_digits
```

Questions to investigate:

1. Does TabPFN still perform well?
2. Does it help more on small training sets?
3. Does it struggle when there are many classes?
4. How does it compare to random forest?
5. Which examples are most uncertain?

In [ ]:
# OPTIONAL TODO:
# Load another dataset and repeat the workflow.